# Notebook 02 — Ingestion bronze (multi-usines + horodatage)

## Objectif

À partir du dataset raw AI4I, on produit le **niveau bronze** de l'architecture médaille MECHA :

- Split du dataset en 2 usines (FR-01, ES-01)
- Ajout d'un **horodatage simulé** (10 000 pièces étalées sur 30 jours)
- Application d'un **léger décalage contrôlé** sur ES-01 pour simuler une dérive inter-site
- Écriture au format **Parquet** (colonnaire, compressé, typé)

## Rattachement grille MSPR

| Compétence | Critère |
|---|---|
| C3 — Stratégie Big Data | Liste les technologies et les outils adaptés, méthodologie de stockage |
| C1 — Collecter les besoins | Modélise le processus de collecte |


## 1. Imports et configuration

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

np.random.seed(42)
rng = np.random.default_rng(42)

PROJECT_ROOT = Path.cwd().parent
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "ai4i2020.csv"
BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Source : {RAW_CSV}")
print(f"Sortie : {BRONZE_DIR}")

Source : /home/romaric420/MSPR/data/raw/ai4i2020.csv
Sortie : /home/romaric420/MSPR/data/bronze


## 2. Configuration des 2 usines

In [2]:
USINES = {
    "FR-01": {
        "pays": "France",
        "ratio": 0.60,
        "tool_wear_offset": 0,
        "torque_noise_std": 0.0,
    },
    "ES-01": {
        "pays": "Espagne",
        "ratio": 0.40,
        "tool_wear_offset": 8,
        "torque_noise_std": 1.5,
    }
}

FENETRE_JOURS = 30
print(pd.DataFrame(USINES).T)

          pays ratio tool_wear_offset torque_noise_std
FR-01   France   0.6                0              0.0
ES-01  Espagne   0.4                8              1.5


## 3. Chargement du dataset brut

In [3]:
df = pd.read_csv(RAW_CSV)
df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]
print(f"Shape : {df.shape}")
df.head(2)

Shape : (10070, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1122,L48301,L,296.6,307.6,1459.0,40.2,96,0,0,0,0,0,0
1,319,L47498,L,297.8,308.4,1512.0,39.9,181,0,0,0,0,0,0


## 4. Simulation d'un horodatage réaliste

In [4]:
start = pd.Timestamp("2025-10-01 06:00:00")
end = start + pd.Timedelta(days=FENETRE_JOURS)
total_sec = (end - start).total_seconds()

offsets = np.sort(rng.uniform(0, total_sec, size=len(df)))
df["timestamp"] = start + pd.to_timedelta(offsets, unit="s")

print(f"Premier timestamp : {df['timestamp'].min()}")
print(f"Dernier timestamp : {df['timestamp'].max()}")
print(f"Duree             : {(df['timestamp'].max() - df['timestamp'].min()).days} jours")

Premier timestamp : 2025-10-01 06:13:34.975717473
Dernier timestamp : 2025-10-31 05:59:00.436922663
Duree             : 29 jours


## 5. Attribution de l'usine (selon ratios)

In [5]:
names = list(USINES.keys())
ratios = np.array([USINES[n]["ratio"] for n in names])
ratios = ratios / ratios.sum()

df["usine"] = rng.choice(names, size=len(df), p=ratios)

print(df["usine"].value_counts())
print()
print(df["usine"].value_counts(normalize=True).round(3))

usine
FR-01    5938
ES-01    4132
Name: count, dtype: int64

usine
FR-01    0.59
ES-01    0.41
Name: proportion, dtype: float64


## 6. Application du décalage inter-site (dérive contrôlée sur ES-01)

Simule une usine avec des outils plus anciens (usure moyenne + 8 min) et des capteurs moins précis (bruit additionnel sur le couple).


In [6]:
def apply_drift(df_site, cfg):
    out = df_site.copy()
    offset = cfg["tool_wear_offset"]
    noise = cfg["torque_noise_std"]

    # Cast si besoin (les colonnes peuvent etre polluees par des strings N/A)
    out["Tool wear [min]"] = pd.to_numeric(out["Tool wear [min]"], errors="coerce")
    out["Torque [Nm]"]     = pd.to_numeric(out["Torque [Nm]"],     errors="coerce")

    if offset:
        out["Tool wear [min]"] = out["Tool wear [min]"] + offset
    if noise:
        n = len(out)
        out["Torque [Nm]"] = out["Torque [Nm]"] + rng.normal(0, noise, size=n)
    return out

chunks = []
for usine, cfg in USINES.items():
    sub = df[df["usine"] == usine].copy()
    sub = apply_drift(sub, cfg)
    sub["pays"] = cfg["pays"]
    chunks.append(sub)

df_bronze = pd.concat(chunks, ignore_index=True).sort_values("timestamp").reset_index(drop=True)
print(df_bronze.groupby("usine").agg(
    pieces=("UDI", "count"),
    tool_wear_moy=("Tool wear [min]", "mean"),
    torque_moy=("Torque [Nm]", "mean"),
).round(2))

       pieces  tool_wear_moy  torque_moy
usine                                   
ES-01    4132         116.88       40.54
FR-01    5938         107.38       40.54


## 7. Écriture bronze (Parquet)

In [7]:
for usine in USINES:
    sub = df_bronze[df_bronze["usine"] == usine].copy()
    out_path = BRONZE_DIR / f"{usine}.parquet"
    sub.to_parquet(out_path, index=False, compression="snappy")
    print(f"  {usine} : {len(sub):5d} lignes -> {out_path.name} ({out_path.stat().st_size/1024:.1f} Ko)")

  FR-01 :  5938 lignes -> FR-01.parquet (179.0 Ko)
  ES-01 :  4132 lignes -> ES-01.parquet (160.1 Ko)


## 8. Vérification

In [8]:
for f in sorted(BRONZE_DIR.glob("*.parquet")):
    tmp = pd.read_parquet(f)
    print(f"  {f.name}")
    print(f"    shape : {tmp.shape}")
    print(f"    cols  : {list(tmp.columns)}")
    print(f"    ts    : {tmp['timestamp'].min()}  ->  {tmp['timestamp'].max()}")
    print()

  ES-01.parquet
    shape : (4132, 17)
    cols  : ['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'timestamp', 'usine', 'pays']
    ts    : 2025-10-01 06:18:00.648612845  ->  2025-10-31 05:37:56.535931567

  FR-01.parquet
    shape : (5938, 17)
    cols  : ['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'timestamp', 'usine', 'pays']
    ts    : 2025-10-01 06:13:34.975717473  ->  2025-10-31 05:59:00.436922663



## 9. Synthèse

### Choix techniques

| Choix | Justification |
|---|---|
| **Parquet** au lieu de CSV | Format colonnaire, compression native, conserve les types |
| **Snappy** pour la compression | Bon compromis vitesse/ratio (standard industriel) |
| **Split par usine** dans des fichiers séparés | Permet de paralléliser les traitements downstream |
| **Horodatage simulé 30 jours** | Suffisant pour calculer des KPIs journaliers et des tendances hebdomadaires |
| **Décalage contrôlé ES-01** | Démontre la capacité à détecter une dérive inter-site (benchmark usines) |

### Diagramme de flux (ingestion)

```
data/raw/ai4i2020.csv
         │
         ▼
  [ Ingestion Python ]
         │
         ├── ajout timestamp
         ├── split aleatoire selon ratios
         ├── apply_drift sur ES-01
         │
         ▼
data/bronze/
   ├── FR-01.parquet
   └── ES-01.parquet
```

### Suite

- **Notebook 03** : nettoyage bronze → silver (qualité Pandera + corrections)
